In [1]:
# Import module yang disediakan google colab untuk kebutuhan upload file

from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"anwhil","key":"0bb9a213ac59d3711c38581b7b553d08"}'}

In [2]:
# Download kaggle dataset and unzip the file
# !cp kaggle.json ~/.kaggle/

# !chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d antobenedetti/animals
!unzip animals.zip

Output streaming akan dipotong hingga 5000 baris terakhir.
  inflating: animals/train/horse/horse2690.jpg  
  inflating: animals/train/horse/horse2691.jpg  
  inflating: animals/train/horse/horse2692.jpg  
  inflating: animals/train/horse/horse2693.jpg  
  inflating: animals/train/horse/horse2694.jpg  
  inflating: animals/train/horse/horse2695.jpg  
  inflating: animals/train/horse/horse2696.jpg  
  inflating: animals/train/horse/horse2697.jpg  
  inflating: animals/train/horse/horse2698.jpg  
  inflating: animals/train/horse/horse2699.jpg  
  inflating: animals/train/horse/horse27.jpg  
  inflating: animals/train/horse/horse270.jpg  
  inflating: animals/train/horse/horse2701.jpg  
  inflating: animals/train/horse/horse2702.jpg  
  inflating: animals/train/horse/horse2703.jpg  
  inflating: animals/train/horse/horse2704.jpg  
  inflating: animals/train/horse/horse2705.jpg  
  inflating: animals/train/horse/horse2706.jpg  
  inflating: animals/train/horse/horse2707.jpg  
  inflating: 

In [3]:
# Direktori awal untuk train dan test
train_dir = "animals/train"
val_dir = "animals/val"

# Direktori baru untuk dataset gabungan
combined_dir = "animals/dataset"

In [4]:
# Buat direktori baru untuk dataset gabungan
import os
os.makedirs(combined_dir, exist_ok=True)

In [5]:
import shutil
# Salin file dan folder dari train
for category in os.listdir(train_dir):
    category_dir = os.path.join(train_dir, category)
    if os.path.isdir(category_dir):
        shutil.copytree(category_dir, os.path.join(combined_dir, category), dirs_exist_ok=True)

# Salin file dan folder dari val
for category in os.listdir(val_dir):
    category_dir = os.path.join(val_dir, category)
    if os.path.isdir(category_dir):
        shutil.copytree(category_dir, os.path.join(combined_dir, category), dirs_exist_ok=True)

In [6]:
import numpy as np
from PIL import Image
# Membuat kamus yang menyimpan gambar untuk setiap kelas dalam data
lung_image = {}

# Tentukan path sumber train
path = "animals/"
path_sub = os.path.join(path, "dataset")
for i in os.listdir(path_sub):
    lung_image[i] = os.listdir(os.path.join(path_sub, i))


In [7]:
import pandas as pd
# Definisikan path sumber
lung_path = "animals/dataset/"

# Buat daftar yang menyimpan data untuk setiap nama file, path file, dan label dalam data
file_name = []
labels = []
full_path = []

# Dapatkan nama file gambar, path file, dan label satu per satu dengan looping, dan simpan sebagai dataframe
for path, subdirs, files in os.walk(lung_path):
    for name in files:
        full_path.append(os.path.join(path, name))
        labels.append(path.split('/')[-1])
        file_name.append(name)



In [8]:
import cv2
import random
from skimage.transform import rotate
from skimage.exposure import adjust_gamma
from skimage import io
from skimage.util import img_as_ubyte
import numpy as np

def resize_img(img):
    return cv2.resize(img, (150, 150))

def subtle_rotation(img):
    img = resize_img(img)
    sudut = random.randint(-30, 30)
    return rotate(img, sudut, preserve_range=True)

def flip_left_right(img):
    img = resize_img(img)
    return np.fliplr(img)

def blur_image(img):
    img = resize_img(img)
    return cv2.GaussianBlur(img, (5, 5), 0)

def add_brightness(img):
    img = resize_img(img)
    gamma_val = random.uniform(0.7, 1.3)
    return adjust_gamma(img, gamma=gamma_val, preserve_range=True)

def enhance_contrast(img):
    img = resize_img(img)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

In [13]:
transformations = {
    'subtle rotation': subtle_rotation,
    'blurring image': blur_image,
    'add brightness': add_brightness,
    'flip horizontal': flip_left_right,
    'resize_img': resize_img,
    'enhace_contrast': enhance_contrast
}

base_path = "animals/dataset/"
target_folders = os.listdir(base_path)[:5]

# 2. Loop untuk setiap folder
for idx, folder in enumerate(target_folders):
    images_path = os.path.join(base_path, folder)
    augmented_path = images_path

    images = [os.path.join(images_path, im) for im in os.listdir(images_path) if im.endswith(('.jpg', '.png', '.jpeg'))]

    # 3. Tentukan jumlah tambahan gambar
    target_total = 4000
    current_count = len(images)
    images_to_generate = target_total - current_count

    i = 1
    print(f"Memproses {folder}: Butuh {max(0, images_to_generate)} gambar tambahan.")

    # 4. Proses pembuatan gambar augmentasi (DIMASUKKAN KE DALAM INDENTASI LOOP FOR)
    while i <= images_to_generate:
        image_path = random.choice(images)
        try:
            original_image = io.imread(image_path)

            if len(original_image.shape) == 2:
                original_image = cv2.cvtColor(original_image, cv2.COLOR_GRAY2RGB)
            elif original_image.shape[2] == 4:
                original_image = cv2.cvtColor(original_image, cv2.COLOR_RGBA2RGB)

            transformed_image = original_image

            transformation_count = random.randint(1, 3)
            for _ in range(transformation_count):
                key = random.choice(list(transformations))
                transformed_image = transformations[key](transformed_image)

            new_image_path = os.path.join(augmented_path, f"aug_{folder}_{i}.jpg")
            transformed_image = img_as_ubyte(transformed_image)

            cv2.imwrite(new_image_path, cv2.cvtColor(transformed_image, cv2.COLOR_RGB2BGR))
            i += 1
        except Exception as e:
            print(f'Gagal memproses {image_path}: {e}')

Memproses elephant: Butuh 0 gambar tambahan.
Memproses lion: Butuh 0 gambar tambahan.
Memproses horse: Butuh 0 gambar tambahan.
Memproses dog: Butuh 0 gambar tambahan.
Memproses cat: Butuh 0 gambar tambahan.


In [10]:
# Definisikan path sumber
base_path = "animals/dataset/"

# Buat daftar yang menyimpan data untuk setiap nama file, path file, dan label dalam data
file_name = []
labels = []
full_path = []

# Dapatkan nama file gambar, path file, dan label (hanya untuk 5 folder pertama seperti di atas)
target_folders = os.listdir(base_path)[:5]

for folder in target_folders:
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        for name in os.listdir(folder_path):
            if name.endswith(('.jpg', '.png', '.jpeg')):
                full_path.append(os.path.join(folder_path, name))
                labels.append(folder)
                file_name.append(name)

In [14]:
# Panggil variabel mypath yang menampung folder dataset gambar
mypath= 'animals/dataset/'

file_name = []
labels = []
full_path = []
for path, subdirs, files in os.walk(mypath):
    for name in files:
        full_path.append(os.path.join(path, name))
        labels.append(path.split('/')[-1])
        file_name.append(name)

# Memasukkan variabel yang sudah dikumpulkan pada looping di atas menjadi sebuah dataframe agar rapi
df = pd.DataFrame({"path":full_path,'file_name':file_name,"labels":labels})
# Melihat jumlah data gambar pada masing-masing label
df.groupby(['labels']).size()

,0
labels,
cat,4000
dog,4000
elephant,4000
horse,4000
lion,4000


In [15]:
# Variabel yang digunakan pada pemisahan data ini di mana variabel x = data path dan y = data labels
from sklearn.model_selection import train_test_split
X= df['path']
y= df['labels']

# Split dataset awal menjadi data train dan test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=300)

In [16]:
# Menyatukan ke dalam masing-masing dataframe
df_tr = pd.DataFrame({'path':X_train,'labels':y_train,'set':'train'})
df_te = pd.DataFrame({'path':X_test,'labels':y_test,'set':'test'})

In [17]:
# Gabungkan DataFrame df_tr dan df_te
df_all = pd.concat([df_tr, df_te], ignore_index=True)

print('===================================================== \n')
print(df_all.groupby(['set', 'labels']).size(), '\n')
print('===================================================== \n')

# Cek sampel data
print(df_all.sample(5))

# Memanggil dataset asli yang berisi keseluruhan data gambar yang sesuai dengan labelnya
datasource_path = "animals/dataset/"
# Membuat variabel Dataset, tempat menampung data yang telah dilakukan pembagian data training dan testing
dataset_path = "Dataset-Final/"


set    labels  
test   cat          818
       dog          811
       elephant     751
       horse        815
       lion         805
train  cat         3182
       dog         3189
       elephant    3249
       horse       3185
       lion        3195
dtype: int64 


                                     path labels    set
3965   animals/dataset/horse/horse128.png  horse  train
15500       animals/dataset/cat/cat59.jpg    cat  train
3984     animals/dataset/lion/lion711.jpg   lion  train
17597    animals/dataset/lion/lion410.jpg   lion   test
5743   animals/dataset/horse/horse294.jpg  horse  train


In [18]:
from tqdm.notebook import tqdm as tq
for index, row in tq(df_all.iterrows()):
    # Deteksi filepath
    file_path = row['path']
    if os.path.exists(file_path) == False:
            file_path = os.path.join(datasource_path,row['labels'],row['image'].split('.')[0])

    # Buat direktori tujuan folder
    if os.path.exists(os.path.join(dataset_path,row['set'],row['labels'])) == False:
        os.makedirs(os.path.join(dataset_path,row['set'],row['labels']))

    # Tentukan tujuan file
    destination_file_name = file_path.split('/')[-1]
    file_dest = os.path.join(dataset_path,row['set'],row['labels'],destination_file_name)

    # Salin file dari sumber ke tujuan
    if os.path.exists(file_dest) == False:
        shutil.copy2(file_path,file_dest)

0it [00:00, ?it/s]

In [19]:
!zip -r hasil_folder.zip /content/Dataset-Final/

Output streaming akan dipotong hingga 5000 baris terakhir.
  adding: content/Dataset-Final/train/dog/dog2086.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/aug_dog_376.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/dog1421.jpg (deflated 0%)
  adding: content/Dataset-Final/train/dog/dog138.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/dog1770.jpg (deflated 0%)
  adding: content/Dataset-Final/train/dog/dog472.jpg (deflated 0%)
  adding: content/Dataset-Final/train/dog/dog1328.jpg (deflated 0%)
  adding: content/Dataset-Final/train/dog/dog2199.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/aug_dog_1365.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/aug_dog_945.jpg (deflated 2%)
  adding: content/Dataset-Final/train/dog/dog1211.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/aug_dog_1248.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog/dog1271.jpg (deflated 1%)
  adding: content/Dataset-Final/train/dog